# How to Use the Benson Increments Calculator

This tool estimates the standard heat of formation for organic molecules using the Benson Group Increment method. Follow these steps:

1. **Prepare the Input File:**
   - Place a file named `increment_correction_table.csv` in the same folder as this notebook.
   - The CSV should have the following columns:
     - `CH Benson Group Increment`: Names of CH group increments
     - `Delta_Hf kJ/mol`: Corresponding increment values (numeric, in kJ/mol)
     - `CHO Benson Group Increment`: Names of CHO group increments
     - `CHO Values`: Corresponding increment values (numeric, in kJ/mol)
     - `Correction`: Names of correction increments
     - `Correction Values`: Corresponding correction values (numeric, in kJ/mol)

2. **Using the Calculator:**
   - Click on the buttons in each tab to add increments or corrections for your molecule.
   - The total heat of formation is updated automatically in both kJ/mol and kcal/mol.
   - Use the Undo button to remove the last increment added.

3. **Output:**
   - The displayed totals represent the estimated standard heat of formation for your selected increments.
   - The pressed buttons list shows your current selection history.

**Note:**
- Ensure all values in the CSV are numeric for correct calculations.
- If the file or columns are missing, the tool will not function as expected.


<h1>Heat of Formation Calculator Using Benson Group Increments</h1>
<h4>Designed for Chemistry C450/C540 at Indiana University Bloomington by Prashant Kumar and Nicola L. B. Pohl</h4><br>
September 12, 2024 Version<br>
<ul>
<li>This notebook is designed to calculate approximate heats of formation of organic molecules based on the idea of Benson Group Increments: Cohen & Benson, <em>Chem. Rev.</em> <b>1993</b>, <em>93</em>, 2419.</li>
<li>The notebook takes values from a user's file titled "increment_table.csv" that is in the same folder as this notebook file. This csv file contains one column titled "Benson Group Increment" populated by Benson Group increment types and a second column titled "Delta_Hf kcal/mol" populated by numerical values in kcal/mol. Research is ongoing to update values and add increments to better describe a range of molecules; the user can decide which increments to use in the calculator.</li>
<li>The Benson Group Increments are rendered into compact buttons that the user can click on to select. The value associated with that button is automatically added to the total displayed below the buttons. The user needs to decide which increments and corrections are needed based on the molecule of interest to make the entire process of estimation transparent.</li> 

In [19]:
# --- Constants ---
# Conversion factor from kilojoules to kilocalories (source: NIST)
KJ_TO_KCAL = 0.239006

# --- Data Loading ---
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# Load data from the CSV file
# Reads the increment and correction table for Benson Group Increments
# Expects columns: 'CH Group', 'Delta_Hf kJ/mol', 'CHO Group', 'CHO Value', 'Correction', 'Correction Value'
data_frame = pd.read_csv('increment_correction_table.csv')

# Clean column names by stripping any leading/trailing spaces
data_frame.columns = data_frame.columns.str.strip()

# Fill NaN values with a default value (e.g., an empty string or zero)
data_frame.fillna(0, inplace=True)

# --- Widget Value Dictionaries ---
# Create dictionaries for the button values
ch_group_values = dict(zip(data_frame['CH Benson Group Increment'], data_frame['Delta_Hf kJ/mol']))
cho_group_values = dict(zip(data_frame['CHO Benson Group Increment'], data_frame['CHO Values']))
correction_values = dict(zip(data_frame['Correction'], data_frame['Correction Values']))

# --- State Variables ---
# Variable to store the total value
# total_heat_kj: running total in kJ/mol
# last_increment_kj: last increment added (for undo)
# selected_button_labels: list of pressed button labels
total_heat_kj = 0.0
last_increment_kj = 0.0  # Store the last added value for undo purposes
selected_button_labels = []  # List to store pressed buttons

# --- UI Labels ---
# Create labels to display the standard heat of formation in kJ/mol and kcal/mol
total_label_kj = widgets.Label(value=f"Standard heat of formation: {total_heat_kj:.2f} kJ/mol")
total_label_kcal = widgets.Label(value=f"Standard heat of formation: {total_heat_kj * KJ_TO_KCAL:.2f} kcal/mol")
selected_buttons_label = widgets.Label(value="Pressed buttons: None")

# --- Event Handlers ---
def add_to_total(value, label):
    """Add the selected increment value to the total and update UI labels."""
    global total_heat_kj, last_increment_kj
    total_heat_kj += value
    last_increment_kj = value  # Store the last value added
    selected_button_labels.append(label)  # Add the button label to the list
    total_label_kj.value = f"Standard heat of formation: {total_heat_kj:.2f} kJ/mol"
    total_label_kcal.value = f"Standard heat of formation: {total_heat_kj * KJ_TO_KCAL:.2f} kcal/mol"
    selected_buttons_label.value = f"Pressed buttons: {', '.join(selected_button_labels)}"

def undo_last_action(button):
    """Undo the last increment addition and update UI labels."""
    global total_heat_kj, last_increment_kj
    if selected_button_labels:
        total_heat_kj -= last_increment_kj  # Subtract the last added value
        selected_button_labels.pop()  # Remove the last pressed button
        total_label_kj.value = f"Standard heat of formation: {total_heat_kj:.2f} kJ/mol"
        total_label_kcal.value = f"Standard heat of formation: {total_heat_kj * KJ_TO_KCAL:.2f} kcal/mol"
        selected_buttons_label.value = f"Pressed buttons: {', '.join(selected_button_labels) if selected_button_labels else 'None'}"
        last_increment_kj = 0.0  # Reset last_value after undoing

# --- Widget Creation ---
def create_button(label, value, color='lightblue'):
    """Create a compact button for a group increment or correction."""
    if label == '' or value == '':
        return None  # Skip creating a button if label or value is empty
    button = widgets.Button(description=f"{label}", layout=widgets.Layout(width="125px", height="25px", padding="0px 0px 0px 0px"))
    button.style.button_color = color
    button.on_click(lambda b: add_to_total(value, label))
    return button

# Create compact buttons for each chemical notation from the CSV file
ch_group_buttons = [create_button(ch_group, value) for ch_group, value in ch_group_values.items() if ch_group and value]
cho_group_buttons = [create_button(cho_group, value, color='lightpink') for cho_group, value in cho_group_values.items() if cho_group and value]
correction_buttons = [create_button(correction, value, color='lightgreen') for correction, value in correction_values.items() if correction and value]

# --- Layouts ---
# Create a compact grid layout for the buttons (e.g., 6 columns)
ch_group_layout = widgets.GridBox([btn for btn in ch_group_buttons if btn], layout=widgets.Layout(grid_template_columns="repeat(6, 150px)", grid_gap="2px"))
cho_group_layout = widgets.GridBox([btn for btn in cho_group_buttons if btn], layout=widgets.Layout(grid_template_columns="repeat(6, 150px)", grid_gap="2px"))
correction_layout = widgets.GridBox([btn for btn in correction_buttons if btn], layout=widgets.Layout(grid_template_columns="repeat(6, 150px)", grid_gap="2px"))

# --- Undo Button ---
# Create the Undo button to allow user to remove the last increment added to the total value
undo_button = widgets.Button(description="Undo the last addition", layout=widgets.Layout(width="900px", height="25px"))
undo_button.on_click(undo_last_action)

# --- Tabbed Interface ---
# Create a tabbed interface for button groups
tab = widgets.Tab()
tab.children = [ch_group_layout, cho_group_layout, correction_layout]
tab.set_title(0, 'CH Groups')
tab.set_title(1, 'CHO Groups')
tab.set_title(2, 'Corrections')

# --- Display UI ---
display(tab)
display(undo_button)
display(total_label_kj)
display(total_label_kcal)
display(selected_buttons_label)


Button(description='Undo the last addition', layout=Layout(height='25px', width='900px'), style=ButtonStyle())

Label(value='Standard heat of formation: 0.00 kJ/mol')

Label(value='Standard heat of formation: 0.00 kcal/mol')

Label(value='Pressed buttons: None')